# L0 step 2 · Our 7B at full size on all six tasks

Task **L0**, step 2, from `Essential_task_delegation/Next_Steps_Roadmap.md`.
- Decided 19 Oct 2026: a top-up training at full size, with both models judged on every optical question.
- Decided 20 Oct: the old model's re-score runs first, on account 3; account 2 joins after R11s7.

Built as a copy of `l0-step1-image-size-check.ipynb`, which stays unchanged, with H1 V2's prompts for the other four tasks. The model loading, greedy decoding and scorer are the same; only the tasks and the model change.

**Why**
- L0 step 1: at full size our 7B gains on disaster type (71.1 → 75.6) and relational reasoning (23.3 → 31.1), but still trails the paper's fine-tune (83.6 and 36.2). Full size is 1036 px for a 1024 px image: Qwen2.5-VL's default and the paper's setting.
- Step 2 trains our adapter a little more at full size. Its keep rule compares the two models on all six tasks at the same size.
- So the old model needs full-size answers on the four tasks step 1 didn't ask, and the new model needs them on all six.

**What it does**
- `MODEL = "old"` is our CSE499A 7B, merged. `MODEL = "new"` is L0 step 2's top-up model, once its training notebook has merged and pushed it.
- It asks the optical questions of the tasks in `RUN_TASKS` at full size (the native image), with greedy decoding, the official prompts and the 499A scorer, as in L0 step 1.

| Task | Optical questions | Images per question | Time at full size |
|---|---|---|---|
| DTR, disaster type | 353 | before + after | 0.5 h |
| BBR, bearing bodies | 2,006 | before + after | 2.6 h |
| BDC, building counting | 3,583 | before + after | 4.7 h |
| DRE, road damage | 1,868 | before + after | 2.4 h |
| DSR, land use | 2,007 | before | 1.3 h |
| ORR, relational | 858 | one image with boxes | 0.6 h |
| All six | 10,675 | | 12.1 h |

The times come from L0 step 1: 4.7 s per two-image question and 2.35 s per one-image question.

**Parts**, so two accounts can share a model's questions:

| Model | Part A | Part B |
|---|---|---|
| old | `["BDC", "BBR"]`: about 7.3 h, account 3 | `["DRE", "DSR"]`: about 3.8 h, account 2 after R11s7 |
| new | `["BDC", "BBR"]`: about 7.3 h | `["DRE", "DSR", "DTR", "ORR"]`: about 4.8 h |

- The old model's DTR and ORR answers come from L0 step 1.
- Each part scores the tasks it has. The full table and the verdict come from all parts' answer files together, on your PC, because one Kaggle account can't attach another account's private output.

**Decision** (fixed 19 Oct, before any run): keep the new model only if disaster type and relational reasoning rise, and no other task drops by more than 1 point, against the old model at full size. The paired difference per task, with its 95% interval, is reported alongside.

**Expectation for the old model's re-score** (fixed 20 Oct, before the run): no decision hangs on it; it's the baseline.
- The four tasks should move by a few points at most from their 512 px scores, since the model trained at 616 px.
- Counting (BDC, 32.0% at 512 px) has the most to gain, because small buildings become visible. Above the base model's 34.2%, the counting half of KR2 is met.

**How to run on Kaggle**
1. Settings: **GPU T4 ×2**, internet ON.
2. Input: `redwannewaz/disasterm3-bench-mirror-v2` (the benchmark JSON and images, found by file name).
3. Secret `HF_TOKEN` (AbrarAlam): optional for the old model, whose repo is public.
4. In Cell 2 set `MODEL` and `RUN_TASKS` for your part. Run with `SMOKE_TEST = True` first: 10 questions per task, about 15 minutes including the 16 GB model download. Then set it to `False` and use Save & Run All.
   - Answers are written as they come.
   - The run stops cleanly at 11 hours. To continue, attach that version's output and run again: answers already given are skipped.
5. Outputs in `/kaggle/working/L0S2/`:
   - `l0_step2_<model>_results.csv` (accuracy per task), `L0_step2_<model>_per_question.csv`, `config.json`, `README.md`;
   - `responses/<model>/<subset>.jsonl`, with question IDs as in H1 V2 and L0 step 1.

In [1]:
# ── Cell 1: Environment (Kaggle's own torch; transformers only if too old for Qwen2.5-VL) ──
import importlib.metadata as _md
import subprocess
import sys

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

def _older(have, want):
    nums = lambda v: tuple(int(x) for x in v.split(".")[:2] if x.isdigit())
    return nums(have) < nums(want)

if _older(_version("transformers"), "4.49") or _version("accelerate") == "0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.49", "accelerate"], check=True)

import torch
import transformers
print(f"✓ torch {torch.__version__}, transformers {transformers.__version__}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: the 7B model in fp16 needs both T4s"

✓ torch 2.11.0+cu128, transformers 5.16.1, python 3.13.15
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration, the benchmark's optical questions and their images ──
import glob
import json
import os
import re
import shutil
import time
from collections import Counter
from pathlib import Path

import pandas as pd

MODELS = {"old": "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP",         # our fine-tuned 7B (CSE499A), merged
          "new": "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP"}    # L0 step 2's top-up, merged by its training notebook
MODEL = "old"                  # the model this session asks
RUN_TASKS = ["BDC", "BBR"]     # this session's part (see the first cell)
OUT_DIR = Path("/kaggle/working/L0S2")
SMOKE_TEST = False             # True = 10 questions per task
BATCH = 8                      # questions per generate() call; halved automatically on out-of-memory
MAX_NEW_TOKENS = 32            # answers are option letters, e.g. "B, E, G"
TIME_LIMIT_H = 11.0            # stop cleanly before Kaggle's 12-hour limit
# (code, subset name used by the 499A scorer, task name in the benchmark, images per question), in H1 V2's order
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition", 2),
         ("BBR", "bearing_body", "Disaster Bearing Bodies Recognition", 2),
         ("BDC", "building_damage_counting", "Building Damage Counting", 2),
         ("DRE", "road_damage_counting", "Road Damage Counting", 2),
         ("DSR", "landuse", "Disaster Scene Recognition", 1),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning", 1)]
EXPECTED_QUESTIONS = {"DTR": 353, "BBR": 2006, "BDC": 3583, "DRE": 1868, "DSR": 2007, "ORR": 858}   # 10,675 optical
REF_OLD_512 = {"DTR": 71.10, "BBR": 19.54, "BDC": 32.01, "DRE": 32.87, "DSR": 37.87, "ORR": 23.31}  # H1 V2, V2b: greedy, 512 px
REF_OLD_FULL = {"DTR": 75.64, "ORR": 31.12}                                    # L0 step 1: the old model at full size
REF_PAPER_FT = {"DTR": 83.6, "BBR": 21.5, "BDC": 34.3, "DRE": 29.4, "DSR": 37.7, "ORR": 36.2}   # DisasterM3 Table 2 (optical)
REF_BASE = {"DTR": 66.6, "BBR": 4.7, "BDC": 34.2, "DRE": 29.3, "DSR": 28.3, "ORR": 23.9}        # the same table, base 7B

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def sensor(e):
    """H1 V2's rule for the after-image's sensor: most entries have post_image_type, ORR entries image_type."""
    return e.get("post_image_type") or e.get("image_type") or "Optical"

assert MODEL in MODELS, f"MODEL must be one of {list(MODELS)}"
unknown = set(RUN_TASKS) - {code for code, *_ in TASKS}
assert RUN_TASKS and not unknown, f"RUN_TASKS takes codes from {[code for code, *_ in TASKS]}; unknown: {unknown}"

with open(find_one("benchmark_release.json", "attach redwannewaz/disasterm3-bench-mirror-v2"), encoding="utf-8") as f:
    bench = json.load(f)

bench_index = {}                # every benchmark image by file name: mirrors nest them differently
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)

questions = {}                  # code -> [(id as in H1 V2 and the 499A scorer, entry)], optical only
for code, subset, task, _ in TASKS:
    qs = [(f"{subset}_{i}", e) for i, e in enumerate(e for e in bench if e.get("task") == task)]
    qs = [(qid, e) for qid, e in qs if sensor(e) == "Optical"]
    questions[code] = qs[:10] if SMOKE_TEST else qs
if not SMOKE_TEST:
    assert {c: len(q) for c, q in questions.items()} == EXPECTED_QUESTIONS, "not the optical questions H1 V2 scored"

needed = {name_of(e[k]) for code in RUN_TASKS for _, e in questions[code]
          for k in ("pre_image_path", "post_image_path", "image_path") if e.get(k)}
found = sum(n in bench_index for n in needed)
if found < 0.99 * len(needed):
    raise FileNotFoundError(f"Only {found:,} of {len(needed):,} benchmark images found: attach the bench mirror.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
hours = sum(len(questions[c]) * (4.7 if n == 2 else 2.35) for c, _, _, n in TASKS if c in RUN_TASKS) / 3600
print(f"✓ L0 step 2{'  [SMOKE TEST]' if SMOKE_TEST else ''}: model '{MODEL}' ({MODELS[MODEL]})")
print(f"  This part: {RUN_TASKS}, {sum(len(questions[c]) for c in RUN_TASKS):,} questions, about {hours:.1f} h")
print(f"  Optical questions per task: {({c: len(q) for c, q in questions.items()})}")
print(f"  Images found: {found:,} of {len(needed):,}")

✓ L0 step 2: model 'old' (AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP)
  This part: ['BDC', 'BBR'], 5,589 questions, about 7.3 h
  Optical questions per task: {'DTR': 353, 'BBR': 2006, 'BDC': 3583, 'DRE': 1868, 'DSR': 2007, 'ORR': 858}
  Images found: 4,334 of 4,334


In [3]:
# ── Cell 3: The official prompts (H1 V2's), the full-size images and the 499A scorer ──
from PIL import Image

# Copied from H1 V2, which copied them from temp_junjue/pyscripts/run_vllm.py (the DisasterM3 authors' runner)
PROMPTS = dict(
    bearing_body="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option(s) from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    landuse="""Analyze the image to answer the following question. Choose the best option(s) from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    single_choice="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
    relational_reasoning_qa="""Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
)

def build(e, subset):
    """-> (chat messages, PIL images in prompt order), built as the official runner builds them. The images stay
    native, so the processor takes them at Qwen2.5-VL's default size: 1036 px for a 1024 px image."""
    if subset == "landuse":
        text, paths = PROMPTS["landuse"].format(prompt=e["prompts"], options_str=e["options_str"]), [e["pre_image_path"]]
    elif subset == "relational_reasoning_qa":     # this task's entries spell it option_str
        text, paths = PROMPTS[subset].format(prompt=e["prompts"], options_str=e["option_str"]), [e["image_path"]]
    else:
        key = "bearing_body" if subset == "bearing_body" else "single_choice"
        text = PROMPTS[key].format(prompt=e["prompts"], options_str=e["options_str"])
        paths = [e["pre_image_path"], e["post_image_path"]]
    parts = text.split("<image>")
    if len(parts) == 1:                            # single-image templates: the image comes first
        content = [{"type": "image"}, {"type": "text", "text": text}]
    else:
        content = [{"type": "text", "text": parts[0]}, {"type": "image"}, {"type": "text", "text": parts[1]},
                   {"type": "image"}, {"type": "text", "text": parts[2]}]
    images = [Image.open(bench_index[name_of(p)]).convert("RGB") for p in paths]
    return [{"role": "user", "content": content}], images

def letters(text):
    """The 499A scorer (4_score_results.py): the set of capital letters left after removing
    'THE ANSWER IS' and 'OPTION'. An answer counts only if this set equals the truth's set."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

# Self-checks: the scorer, and the number of images in every task's prompt
assert letters("B, E, G") == {"B", "E", "G"} and letters("The answer is D.") == {"D"} and letters("") == set()
for code, subset, _, n_images in TASKS:
    assert len(build(questions[code][0][1], subset)[1]) == n_images, f"{code}: expected {n_images} image(s)"
native = Counter(img.size for code, subset, _, _ in TASKS if code in RUN_TASKS
                 for _, e in questions[code][:20] for img in build(e, subset)[1])
print(f"✓ Prompts ready (self-checks passed). Native sizes of this part's first 20 questions' images: {dict(native)}")

✓ Prompts ready (self-checks passed). Native sizes of this part's first 20 questions' images: {(1024, 1024): 80}


In [4]:
# ── Cell 4: Load the model (fp16, both T4s), check the full size, and the batched greedy generator ──
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODELS[MODEL], token=hf_token())
processor.tokenizer.padding_side = "left"              # batched generation needs left padding
# transformers 4.56 renamed torch_dtype to dtype; older versions may ignore dtype and load fp32 (30 GB)
dtype_arg = "torch_dtype" if _older(transformers.__version__, "4.56") else "dtype"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODELS[MODEL], **{dtype_arg: torch.float16}, device_map="auto", attn_implementation="sdpa", token=hf_token())
model.eval()

# Full size on a 1024 x 1024 test image, from the processor's grid of 14 px patches. Both merged repos keep the
# stock Qwen2.5-VL processor (1_merge_and_push_model.ipynb and L0 step 2's training notebook), so nothing caps it.
_, h, w = processor.image_processor(images=[Image.new("RGB", (1024, 1024), (90, 120, 80))],
                                    return_tensors="pt")["image_grid_thw"][0].tolist()
assert (h * 14, w * 14) == (1036, 1036), (f"The model would see {h * 14} x {w * 14} px, not 1036: "
                                          "check the processor's min_pixels and max_pixels.")

fed = Counter()                 # image tokens fed and questions asked

@torch.inference_mode()
def generate(batch):
    """batch: list of (messages, images) -> list of answers. Greedy, so a re-run gives the same answers."""
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m, _ in batch]
    images = [img for _, imgs in batch for img in imgs]
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None,
                         top_p=None, top_k=None, repetition_penalty=1.0)
    fed["image_tokens"] += int(inputs["image_grid_thw"].prod(1).sum()) // 4
    fed["questions"] += len(batch)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return [s.strip() for s in processor.batch_decode(new_tokens, skip_special_tokens=True)]

def generate_safe(batch):
    """Halves the batch on out-of-memory instead of crashing the run."""
    try:
        return generate(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            return [""]
        half = len(batch) // 2
        return generate_safe(batch[:half]) + generate_safe(batch[half:])

subset_of = {code: subset for code, subset, _, _ in TASKS}
for code in RUN_TASKS:          # one question of each task in this part, before the long run
    e = questions[code][0][1]
    print(f"  {code}: truth {e['ground_truth_option']!r} | answer {generate_safe([build(e, subset_of[code])])[0]!r}")
fed.clear()
print(f"✓ Model '{MODEL}' loaded ({dtype_arg}=float16), full size checked (1036 px), in {(time.time() - t0) / 60:.1f} min; "
      f"GPU memory {[f'{torch.cuda.memory_allocated(i) / 1e9:.1f} GB' for i in range(torch.cuda.device_count())]}")

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.78k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

  BDC: truth 'A' | answer 'C'
  BBR: truth 'B, E, F, G.' | answer 'B, E, F, G'
✓ Model 'old' loaded (dtype=float16), full size checked (1036 px), in 2.3 min; GPU memory ['7.6 GB', '9.0 GB']


## Run: this part's questions
Tasks run in the order of `RUN_TASKS`. Answers are appended to `responses/<model>/<subset>.jsonl` after every batch; a re-run (or a new version with this one's output attached) skips the questions already answered.

In [5]:
# ── Cell 5: Ask this part's questions, writing answers as they come ──
from tqdm.auto import tqdm

# Continue an earlier run: copy its answers in when that version's output is attached
previous = sorted(glob.glob("/kaggle/input/**/L0S2/responses", recursive=True))
if previous and not (OUT_DIR / "responses").exists():
    shutil.copytree(previous[0], OUT_DIR / "responses")
    print(f"Continuing from {previous[0]}")

t_start = time.time()
stopped, unreadable, minutes = False, Counter(), {}
for code in RUN_TASKS:
    path = OUT_DIR / "responses" / MODEL / f"{subset_of[code]}.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    done = {json.loads(line)["id"] for line in open(path, encoding="utf-8")} if path.exists() else set()
    todo = [(qid, e) for qid, e in questions[code] if qid not in done]
    t_task = time.time()
    with open(path, "a", encoding="utf-8") as f:
        for i in tqdm(range(0, len(todo), BATCH), desc=f"{MODEL} {code}"):
            if (time.time() - t_start) / 3600 > TIME_LIMIT_H:
                stopped = True
                break
            built = []
            for qid, e in todo[i:i + BATCH]:
                try:
                    built.append((qid, build(e, subset_of[code])))
                except (KeyError, OSError):              # image missing or unreadable: no answer, scored wrong
                    unreadable[code] += 1
                    f.write(json.dumps({"id": qid, "response": ""}) + "\n")
            answers = generate_safe([b for _, b in built]) if built else []
            for (qid, _), answer in zip(built, answers):
                f.write(json.dumps({"id": qid, "response": answer}) + "\n")
            f.flush()
    minutes[code] = round((time.time() - t_task) / 60, 1)
    if stopped:
        break
run_hours = (time.time() - t_start) / 3600
print(f"{'⚠ Stopped at the time limit' if stopped else '✓ All questions answered'} after {run_hours:.1f} h; "
      f"minutes per task: {minutes}; unreadable images: {dict(unreadable) or 'none'}")

old BDC:   0%|          | 0/448 [00:00<?, ?it/s]

old BBR:   0%|          | 0/251 [00:00<?, ?it/s]

✓ All questions answered after 8.6 h; minutes per task: {'BDC': 336.3, 'BBR': 180.8}; unreadable images: {'BDC': 3, 'BBR': 2}


In [6]:
# ── Cell 6: Score this model's answers here against the references, and the output files ──
answers = {}
for path in (OUT_DIR / "responses" / MODEL).glob("*.jsonl"):
    for line in open(path, encoding="utf-8"):
        r = json.loads(line)
        answers[r["id"]] = r["response"]

rows = []
for code, _, _, _ in TASKS:
    for qid, e in questions[code]:
        rows.append({"task": code, "id": qid, "truth": e["ground_truth_option"], "answer": answers.get(qid),
                     "correct": letters(answers[qid]) == letters(e["ground_truth_option"]) if qid in answers else None})
per_q = pd.DataFrame(rows)
per_q.to_csv(OUT_DIR / f"L0_step2_{MODEL}_per_question.csv", index=False)

table = []
for code, _, _, _ in TASKS:
    got = per_q.loc[per_q["task"] == code, "correct"].dropna().astype(bool)
    table.append({"task": code, "questions": len(questions[code]), "answered": len(got),
                  f"{MODEL}_full_%": round(100 * got.mean(), 2) if len(got) else float("nan"),
                  "old_512_%": REF_OLD_512[code], "old_full_L0S1_%": REF_OLD_FULL.get(code, float("nan")),
                  "paper_finetuned_%": REF_PAPER_FT[code], "paper_base_%": REF_BASE[code]})
results = pd.DataFrame(table)
results.to_csv(OUT_DIR / f"l0_step2_{MODEL}_results.csv", index=False)
print(results.to_string(index=False))

complete = [r["task"] for r in table if r["answered"] == r["questions"]]
notes = []
if "BDC" in complete and not SMOKE_TEST:
    bdc = next(r[f"{MODEL}_full_%"] for r in table if r["task"] == "BDC")
    notes.append(f"Building counting at full size: {bdc}% against the base model's 34.2%: the counting half of KR2 is "
                 + ("met." if bdc > 34.2 else "still not met."))
print("\n".join(["", *notes]))

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

tokens = round(fed["image_tokens"] / fed["questions"]) if fed["questions"] else None
config = {"task": "L0 step 2, full-size re-score", "smoke_test": SMOKE_TEST, "model": MODEL, "model_id": MODELS[MODEL],
          "run_tasks": RUN_TASKS, "decoding": "greedy", "image_size": "native (1036 px for a 1024 px image)",
          "image_tokens_per_question": tokens, "questions": {c: len(q) for c, q in questions.items()},
          "complete_tasks": complete, "batch": BATCH, "max_new_tokens": MAX_NEW_TOKENS,
          "transformers": transformers.__version__, "torch": torch.__version__, "stopped_at_time_limit": stopped,
          "run_hours": round(run_hours, 2), "minutes": minutes, "unreadable_images": dict(unreadable)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    f"# L0 step 2: the {MODEL} 7B at full size", "",
    f"Model `{MODELS[MODEL]}`, greedy decoding, the official DisasterM3 prompts, optical questions only, native images "
    f"(1036 px for a 1024 px image){' (SMOKE TEST)' if SMOKE_TEST else ''}. This session asked {RUN_TASKS}; "
    f"complete tasks here: {complete}.", "",
    "## Accuracy (%)", "", as_markdown(results), "",
    *[f"**{n}**" for n in notes], "",
    "## Notes", "",
    "- `old_512_%`: H1 V2 and V2b (greedy, 512 px, the same optical questions). `old_full_L0S1_%`: L0 step 1 at full size.",
    "- The paper's columns come from DisasterM3's Table 2 (optical questions), each model at Qwen's default image size.",
    "- The keep rule (fixed 19 Oct) is applied on your PC, to all parts' answers together: the new model is kept only if "
    "disaster type and relational reasoning rise and no other task drops by more than 1 point against the old model "
    "at full size.", "",
    "## Files", "",
    f"- `l0_step2_{MODEL}_results.csv`, `L0_step2_{MODEL}_per_question.csv`, `config.json`",
    f"- `responses/{MODEL}/<subset>.jsonl`, with question IDs as in H1 V2 and L0 step 1",
]), encoding="utf-8")
print(f"✓ Wrote l0_step2_{MODEL}_results.csv, L0_step2_{MODEL}_per_question.csv, README.md and config.json to {OUT_DIR}")

task  questions  answered  old_full_%  old_512_%  old_full_L0S1_%  paper_finetuned_%  paper_base_%
 DTR        353         0         NaN      71.10            75.64               83.6          66.6
 BBR       2006      2006       20.94      19.54              NaN               21.5           4.7
 BDC       3583      3583       30.09      32.01              NaN               34.3          34.2
 DRE       1868         0         NaN      32.87              NaN               29.4          29.3
 DSR       2007         0         NaN      37.87              NaN               37.7          28.3
 ORR        858         0         NaN      23.31            31.12               36.2          23.9

Building counting at full size: 30.09% against the base model's 34.2%: the counting half of KR2 is still not met.
✓ Wrote l0_step2_old_results.csv, L0_step2_old_per_question.csv, README.md and config.json to /kaggle/working/L0S2
